# 04 — Generate Subscription CDC Events

## Purpose

This notebook generates deterministic subscription Change Data Capture events received after the initial subscription snapshot.

The batch simulates plan upgrades, plan downgrades, cancellations, pauses, and new subscriptions. Customer changes are applied first so that new subscriptions can reference customers introduced by the CRM CDC batch.

## CDC Event Distribution

- 125 plan upgrades
- 75 plan downgrades
- 60 cancellations
- 40 pauses
- 250 new subscriptions
- 550 total events

Cancellations and pauses are represented as UPDATE events because the subscription record remains part of the business history.

## Data Quality Controls

- Customer current-state reconstruction
- Subscription-key validation
- Customer referential integrity
- Deleted-customer validation
- Cross-entity event sequencing
- Duplicate-event detection
- Pricing reconciliation
- Billing-amount reconciliation
- Lifecycle-date validation
- CDC operation validation

## Inputs

- `/Volumes/workspace/revenue_leakage_bronze/landing/crm/customers/initial_load`
- `/Volumes/workspace/revenue_leakage_bronze/landing/crm/customers/change_batch_001`
- `/Volumes/workspace/revenue_leakage_bronze/landing/subscription_system/subscriptions/initial_load`

## Target

`/Volumes/workspace/revenue_leakage_bronze/landing/subscription_system/subscriptions/change_batch_001`

## 1. Configuration and Schemas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    IntegerType,
    DecimalType,
    BooleanType,
)

INITIAL_CUSTOMER_COUNT = 5000
EXPECTED_CURRENT_CUSTOMER_COUNT = 5150
INITIAL_SUBSCRIPTION_COUNT = 6000

UPGRADE_COUNT = 125
DOWNGRADE_COUNT = 75
CANCELLATION_COUNT = 60
PAUSE_COUNT = 40
INSERT_COUNT = 250

EXPECTED_UPDATE_COUNT = (
    UPGRADE_COUNT
    + DOWNGRADE_COUNT
    + CANCELLATION_COUNT
    + PAUSE_COUNT
)

EXPECTED_EVENT_COUNT = EXPECTED_UPDATE_COUNT + INSERT_COUNT

CHANGE_DATE = "2026-08-16"

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/landing"
)

CUSTOMERS_PATH = (
    f"{LANDING_PATH}/crm/customers"
)

CUSTOMERS_INITIAL_PATH = (
    f"{CUSTOMERS_PATH}/initial_load"
)

CUSTOMERS_CHANGE_BATCH_PATH = (
    f"{CUSTOMERS_PATH}/change_batch_001"
)

SUBSCRIPTIONS_PATH = (
    f"{LANDING_PATH}/subscription_system/subscriptions"
)

SUBSCRIPTIONS_INITIAL_PATH = (
    f"{SUBSCRIPTIONS_PATH}/initial_load"
)

SUBSCRIPTIONS_CHANGE_BATCH_PATH = (
    f"{SUBSCRIPTIONS_PATH}/change_batch_001"
)

CUSTOMER_KEY_SCHEMA = StructType([
    StructField("customer_id", StringType(), False),
])

CUSTOMER_CHANGE_SCHEMA = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("country", StringType(), False),
    StructField("region", StringType(), False),
    StructField("customer_segment", StringType(), False),
    StructField("signup_date", DateType(), True),
    StructField("customer_status", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("event_timestamp", TimestampType(), True),
])

SUBSCRIPTION_SCHEMA = StructType([
    StructField("subscription_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("plan_id", StringType(), False),
    StructField("plan_name", StringType(), False),
    StructField("start_date", DateType(), False),
    StructField("end_date", DateType(), True),
    StructField("subscription_status", StringType(), False),
    StructField("billing_frequency", StringType(), False),
    StructField("billing_day", IntegerType(), False),
    StructField(
        "base_monthly_price",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "discount_percentage",
        DecimalType(5, 2),
        False
    ),
    StructField(
        "contracted_monthly_price",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "contracted_billing_amount",
        DecimalType(12, 2),
        False
    ),
    StructField("discount_start_date", DateType(), True),
    StructField("discount_end_date", DateType(), True),
    StructField(
        "included_usage_units",
        IntegerType(),
        False
    ),
    StructField(
        "overage_unit_price",
        DecimalType(10, 4),
        False
    ),
    StructField(
        "payment_terms_days",
        IntegerType(),
        False
    ),
    StructField("auto_renew", BooleanType(), False),
    StructField("currency", StringType(), False),
    StructField("operation", StringType(), False),
    StructField(
        "event_timestamp",
        TimestampType(),
        False
    ),
    StructField("snapshot_date", DateType(), False),
])

SUBSCRIPTION_COLUMNS = SUBSCRIPTION_SCHEMA.fieldNames()

## 2. Reconstruct the Current Customer State

Apply customer INSERT and DELETE events to the initial customer keys. Load the initial subscription snapshot using an explicit schema.

In [0]:
initial_customer_keys_df = (
    spark.read
    .schema(CUSTOMER_KEY_SCHEMA)
    .json(CUSTOMERS_INITIAL_PATH)
    .select("customer_id")
    .distinct()
)

customer_changes_df = (
    spark.read
    .schema(CUSTOMER_CHANGE_SCHEMA)
    .json(CUSTOMERS_CHANGE_BATCH_PATH)
)

inserted_customer_events_df = (
    customer_changes_df
    .filter(F.col("operation") == "INSERT")
    .select(
        "customer_id",
        F.col("event_timestamp").alias(
            "customer_event_timestamp"
        )
    )
)

deleted_customer_keys_df = (
    customer_changes_df
    .filter(F.col("operation") == "DELETE")
    .select("customer_id")
    .distinct()
)

current_customer_keys_df = (
    initial_customer_keys_df
    .join(
        deleted_customer_keys_df,
        on="customer_id",
        how="left_anti"
    )
    .unionByName(
        inserted_customer_events_df
        .select("customer_id")
    )
    .distinct()
)

subscriptions_initial_df = (
    spark.read
    .schema(SUBSCRIPTION_SCHEMA)
    .json(SUBSCRIPTIONS_INITIAL_PATH)
)

current_customer_count = (
    current_customer_keys_df.count()
)

initial_subscription_count = (
    subscriptions_initial_df.count()
)

assert (
    current_customer_count
    == EXPECTED_CURRENT_CUSTOMER_COUNT
)

assert (
    initial_subscription_count
    == INITIAL_SUBSCRIPTION_COUNT
)

print(
    f"Current customer keys: "
    f"{current_customer_count:,}"
)

print(
    f"Initial subscriptions loaded: "
    f"{initial_subscription_count:,}"
)

## 3. Generate Subscription UPDATE Events

Generate mutually exclusive plan upgrades, plan downgrades, cancellations, and pauses. Recalculate all dependent commercial fields after a plan change.

In [0]:
def apply_plan_attributes(subscription_df):
    return (
        subscription_df

        .withColumn(
            "plan_name",
            F.when(
                F.col("plan_id") == "PLAN_STARTER",
                "Starter"
            )
            .when(
                F.col("plan_id") == "PLAN_GROWTH",
                "Growth"
            )
            .when(
                F.col("plan_id") == "PLAN_PROFESSIONAL",
                "Professional"
            )
            .otherwise("Enterprise")
        )

        .withColumn(
            "base_monthly_price",
            F.when(
                F.col("plan_id") == "PLAN_STARTER",
                29.00
            )
            .when(
                F.col("plan_id") == "PLAN_GROWTH",
                79.00
            )
            .when(
                F.col("plan_id") == "PLAN_PROFESSIONAL",
                149.00
            )
            .otherwise(399.00)
            .cast(DecimalType(10, 2))
        )

        .withColumn(
            "included_usage_units",
            F.when(
                F.col("plan_id") == "PLAN_STARTER",
                1000
            )
            .when(
                F.col("plan_id") == "PLAN_GROWTH",
                5000
            )
            .when(
                F.col("plan_id") == "PLAN_PROFESSIONAL",
                15000
            )
            .otherwise(50000)
            .cast("int")
        )

        .withColumn(
            "overage_unit_price",
            F.when(
                F.col("plan_id") == "PLAN_STARTER",
                0.0500
            )
            .when(
                F.col("plan_id") == "PLAN_GROWTH",
                0.0400
            )
            .when(
                F.col("plan_id") == "PLAN_PROFESSIONAL",
                0.0300
            )
            .otherwise(0.0200)
            .cast(DecimalType(10, 4))
        )

        .withColumn(
            "contracted_monthly_price",
            F.round(
                F.col("base_monthly_price")
                * (
                    1
                    - F.col("discount_percentage") / 100
                ),
                2
            ).cast(DecimalType(10, 2))
        )

        .withColumn(
            "contracted_billing_amount",
            F.when(
                F.col("billing_frequency") == "Annual",
                F.col("contracted_monthly_price") * 12
            )
            .otherwise(
                F.col("contracted_monthly_price")
            )
            .cast(DecimalType(12, 2))
        )
    )


active_subscriptions_df = (
    subscriptions_initial_df
    .filter(
        F.col("subscription_status") == "Active"
    )
)

subscription_upgrades_df = (
    active_subscriptions_df
    .filter(
        F.col("plan_id") != "PLAN_ENTERPRISE"
    )
    .orderBy("subscription_id")
    .limit(UPGRADE_COUNT)

    .withColumn(
        "plan_id",
        F.when(
            F.col("plan_id") == "PLAN_STARTER",
            "PLAN_GROWTH"
        )
        .when(
            F.col("plan_id") == "PLAN_GROWTH",
            "PLAN_PROFESSIONAL"
        )
        .otherwise("PLAN_ENTERPRISE")
    )
)

subscription_upgrades_df = (
    apply_plan_attributes(subscription_upgrades_df)
    .withColumn("operation", F.lit("UPDATE"))
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.lit("2026-08-16 11:00:00")
        )
    )
    .withColumn(
        "snapshot_date",
        F.lit(CHANGE_DATE).cast("date")
    )
)

upgrade_keys_df = (
    subscription_upgrades_df
    .select("subscription_id")
)

subscription_downgrades_df = (
    active_subscriptions_df
    .filter(
        F.col("plan_id") != "PLAN_STARTER"
    )
    .join(
        upgrade_keys_df,
        on="subscription_id",
        how="left_anti"
    )
    .orderBy("subscription_id")
    .limit(DOWNGRADE_COUNT)

    .withColumn(
        "plan_id",
        F.when(
            F.col("plan_id") == "PLAN_ENTERPRISE",
            "PLAN_PROFESSIONAL"
        )
        .when(
            F.col("plan_id") == "PLAN_PROFESSIONAL",
            "PLAN_GROWTH"
        )
        .otherwise("PLAN_STARTER")
    )
)

subscription_downgrades_df = (
    apply_plan_attributes(subscription_downgrades_df)
    .withColumn("operation", F.lit("UPDATE"))
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.lit("2026-08-16 11:02:00")
        )
    )
    .withColumn(
        "snapshot_date",
        F.lit(CHANGE_DATE).cast("date")
    )
)

plan_change_keys_df = (
    upgrade_keys_df
    .unionByName(
        subscription_downgrades_df
        .select("subscription_id")
    )
)

subscription_cancellations_df = (
    active_subscriptions_df
    .join(
        plan_change_keys_df,
        on="subscription_id",
        how="left_anti"
    )
    .orderBy("subscription_id")
    .limit(CANCELLATION_COUNT)

    .withColumn(
        "subscription_status",
        F.lit("Cancelled")
    )
    .withColumn(
        "end_date",
        F.lit(CHANGE_DATE).cast("date")
    )
    .withColumn("auto_renew", F.lit(False))
    .withColumn("operation", F.lit("UPDATE"))
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.lit("2026-08-16 11:04:00")
        )
    )
    .withColumn(
        "snapshot_date",
        F.lit(CHANGE_DATE).cast("date")
    )
)

used_update_keys_df = (
    plan_change_keys_df
    .unionByName(
        subscription_cancellations_df
        .select("subscription_id")
    )
)

subscription_pauses_df = (
    active_subscriptions_df
    .join(
        used_update_keys_df,
        on="subscription_id",
        how="left_anti"
    )
    .orderBy("subscription_id")
    .limit(PAUSE_COUNT)

    .withColumn(
        "subscription_status",
        F.lit("Paused")
    )
    .withColumn("auto_renew", F.lit(False))
    .withColumn("operation", F.lit("UPDATE"))
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.lit("2026-08-16 11:06:00")
        )
    )
    .withColumn(
        "snapshot_date",
        F.lit(CHANGE_DATE).cast("date")
    )
)

subscription_updates_df = (
    subscription_upgrades_df
    .unionByName(subscription_downgrades_df)
    .unionByName(subscription_cancellations_df)
    .unionByName(subscription_pauses_df)
    .select(*SUBSCRIPTION_COLUMNS)
)

## 4. Generate New Subscription Events

Generate 250 new subscriptions. Two hundred subscriptions reference customers introduced by the CRM CDC batch, while fifty represent additional subscriptions for existing customers.

In [0]:
new_customer_targets_df = (
    inserted_customer_events_df
    .select("customer_id")
    .distinct()
    .withColumn(
        "insert_sequence",
        F.regexp_extract(
            "customer_id",
            r"(\d+)$",
            1
        ).cast("int")
        - F.lit(INITIAL_CUSTOMER_COUNT)
    )
)

existing_customer_targets_df = (
    current_customer_keys_df
    .join(
        new_customer_targets_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .withColumn(
        "customer_number",
        F.regexp_extract(
            "customer_id",
            r"(\d+)$",
            1
        ).cast("int")
    )
    .filter(
        F.col("customer_number").between(
            1,
            INSERT_COUNT - 200
        )
    )
    .withColumn(
        "insert_sequence",
        F.col("customer_number") + F.lit(200)
    )
    .select(
        "customer_id",
        "insert_sequence"
    )
)

insert_customer_targets_df = (
    new_customer_targets_df
    .unionByName(existing_customer_targets_df)
    .withColumnRenamed(
        "customer_id",
        "target_customer_id"
    )
)

subscription_templates_df = (
    subscriptions_initial_df
    .withColumn(
        "insert_sequence",
        F.regexp_extract(
            "subscription_id",
            r"(\d+)$",
            1
        ).cast("int")
    )
    .filter(
        F.col("insert_sequence").between(
            1,
            INSERT_COUNT
        )
    )
)

subscription_inserts_df = (
    subscription_templates_df
    .join(
        insert_customer_targets_df,
        on="insert_sequence",
        how="inner"
    )
    .withColumn(
        "subscription_id",
        F.format_string(
            "S%07d",
            F.lit(INITIAL_SUBSCRIPTION_COUNT)
            + F.col("insert_sequence")
        )
    )
    .withColumn(
        "customer_id",
        F.col("target_customer_id")
    )
    .withColumn(
        "start_date",
        F.lit(CHANGE_DATE).cast("date")
    )
    .withColumn(
        "end_date",
        F.lit(None).cast("date")
    )
    .withColumn(
        "subscription_status",
        F.lit("Active")
    )
    .withColumn(
        "discount_start_date",
        F.when(
            F.col("discount_percentage") > 0,
            F.lit(CHANGE_DATE).cast("date")
        ).otherwise(
            F.lit(None).cast("date")
        )
    )
    .withColumn(
        "discount_end_date",
        F.when(
            F.col("discount_percentage") > 0,
            F.date_add(
                F.lit(CHANGE_DATE).cast("date"),
                180
            )
        ).otherwise(
            F.lit(None).cast("date")
        )
    )
    .withColumn(
        "auto_renew",
        F.lit(True)
    )
    .withColumn(
        "operation",
        F.lit("INSERT")
    )
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.lit("2026-08-16 11:15:00")
        )
    )
    .withColumn(
        "snapshot_date",
        F.lit(CHANGE_DATE).cast("date")
    )
    .select(*SUBSCRIPTION_COLUMNS)
)

## 5. Combine and Validate the Subscription CDC Batch

Combine UPDATE and INSERT events. Validate event distribution, business keys, customer relationships, event sequencing, pricing rules, and lifecycle dates.

In [0]:
subscription_changes_df = (
    subscription_updates_df
    .unionByName(subscription_inserts_df)
)

upgrade_count = subscription_upgrades_df.count()
downgrade_count = subscription_downgrades_df.count()
cancellation_count = subscription_cancellations_df.count()
pause_count = subscription_pauses_df.count()
insert_count = subscription_inserts_df.count()

actual_event_count = (
    subscription_changes_df.count()
)

distinct_event_subscription_count = (
    subscription_changes_df
    .select("subscription_id")
    .distinct()
    .count()
)

operation_counts = {
    row["operation"]: row["count"]
    for row in (
        subscription_changes_df
        .groupBy("operation")
        .count()
        .collect()
    )
}

null_event_key_count = (
    subscription_changes_df
    .filter(
        F.col("subscription_id").isNull()
        | F.col("customer_id").isNull()
        | F.col("event_timestamp").isNull()
    )
    .count()
)

duplicate_event_count = (
    subscription_changes_df
    .groupBy(
        "subscription_id",
        "operation",
        "event_timestamp"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

initial_subscription_keys_df = (
    subscriptions_initial_df
    .select("subscription_id")
)

missing_update_key_count = (
    subscription_updates_df
    .select("subscription_id")
    .join(
        initial_subscription_keys_df,
        on="subscription_id",
        how="left_anti"
    )
    .count()
)

conflicting_insert_key_count = (
    subscription_inserts_df
    .select("subscription_id")
    .join(
        initial_subscription_keys_df,
        on="subscription_id",
        how="inner"
    )
    .count()
)

orphan_customer_count = (
    subscription_changes_df
    .select("customer_id")
    .distinct()
    .join(
        current_customer_keys_df,
        on="customer_id",
        how="left_anti"
    )
    .count()
)

deleted_customer_subscription_count = (
    subscription_changes_df
    .select("customer_id")
    .join(
        deleted_customer_keys_df,
        on="customer_id",
        how="inner"
    )
    .count()
)

new_customer_subscription_count = (
    subscription_inserts_df
    .select("customer_id")
    .join(
        new_customer_targets_df,
        on="customer_id",
        how="inner"
    )
    .count()
)

subscription_before_customer_event_count = (
    subscription_inserts_df
    .select(
        "customer_id",
        "event_timestamp"
    )
    .join(
        inserted_customer_events_df,
        on="customer_id",
        how="inner"
    )
    .filter(
        F.col("event_timestamp")
        <= F.col("customer_event_timestamp")
    )
    .count()
)

pricing_mismatch_count = (
    subscription_changes_df
    .filter(
        F.abs(
            F.col("contracted_monthly_price")
            - F.round(
                F.col("base_monthly_price")
                * (
                    1
                    - F.col("discount_percentage") / 100
                ),
                2
            )
        ) > 0.01
    )
    .count()
)

billing_amount_mismatch_count = (
    subscription_changes_df
    .filter(
        F.abs(
            F.col("contracted_billing_amount")
            - F.when(
                F.col("billing_frequency") == "Annual",
                F.col("contracted_monthly_price") * 12
            )
            .otherwise(
                F.col("contracted_monthly_price")
            )
        ) > 0.01
    )
    .count()
)

invalid_date_count = (
    subscription_changes_df
    .filter(
        (
            F.col("start_date")
            > F.col("snapshot_date")
        )

        | (
            (
                F.col("subscription_status")
                == "Cancelled"
            )
            & F.col("end_date").isNull()
        )

        | (
            F.col("end_date").isNotNull()
            & (
                F.col("end_date")
                < F.col("start_date")
            )
        )
    )
    .count()
)

max_initial_event_timestamp = (
    subscriptions_initial_df
    .agg(
        F.max("event_timestamp")
        .alias("max_event_timestamp")
    )
    .first()["max_event_timestamp"]
)

min_change_event_timestamp = (
    subscription_changes_df
    .agg(
        F.min("event_timestamp")
        .alias("min_event_timestamp")
    )
    .first()["min_event_timestamp"]
)

assert upgrade_count == UPGRADE_COUNT
assert downgrade_count == DOWNGRADE_COUNT
assert cancellation_count == CANCELLATION_COUNT
assert pause_count == PAUSE_COUNT
assert insert_count == INSERT_COUNT

assert actual_event_count == EXPECTED_EVENT_COUNT

assert (
    distinct_event_subscription_count
    == EXPECTED_EVENT_COUNT
)

assert (
    operation_counts.get("UPDATE")
    == EXPECTED_UPDATE_COUNT
)

assert operation_counts.get("INSERT") == INSERT_COUNT
assert null_event_key_count == 0
assert duplicate_event_count == 0
assert missing_update_key_count == 0
assert conflicting_insert_key_count == 0
assert orphan_customer_count == 0
assert deleted_customer_subscription_count == 0
assert new_customer_subscription_count == 200
assert subscription_before_customer_event_count == 0
assert pricing_mismatch_count == 0
assert billing_amount_mismatch_count == 0
assert invalid_date_count == 0

assert (
    min_change_event_timestamp
    > max_initial_event_timestamp
)

print(f"Plan upgrades: {upgrade_count:,}")
print(f"Plan downgrades: {downgrade_count:,}")
print(f"Cancellations: {cancellation_count:,}")
print(f"Pauses: {pause_count:,}")
print(f"New subscriptions: {insert_count:,}")

print(
    f"Generated subscription events: "
    f"{actual_event_count:,}"
)

print(
    f"Distinct event subscriptions: "
    f"{distinct_event_subscription_count:,}"
)

print(f"Null event keys: {null_event_key_count:,}")
print(f"Duplicate events: {duplicate_event_count:,}")

print(
    f"Missing UPDATE keys: "
    f"{missing_update_key_count:,}"
)

print(
    f"Conflicting INSERT keys: "
    f"{conflicting_insert_key_count:,}"
)

print(
    f"Orphan customer keys: "
    f"{orphan_customer_count:,}"
)

print(
    "Events for deleted customers: "
    f"{deleted_customer_subscription_count:,}"
)

print(
    "New-customer subscriptions: "
    f"{new_customer_subscription_count:,}"
)

print(
    "Subscriptions before customer events: "
    f"{subscription_before_customer_event_count:,}"
)

print(
    f"Pricing mismatches: "
    f"{pricing_mismatch_count:,}"
)

print(
    f"Billing amount mismatches: "
    f"{billing_amount_mismatch_count:,}"
)

print(
    f"Invalid date relationships: "
    f"{invalid_date_count:,}"
)

print(
    "Latest initial subscription event: "
    f"{max_initial_event_timestamp}"
)

print(
    "Earliest subscription change event: "
    f"{min_change_event_timestamp}"
)

display(
    subscription_changes_df
    .groupBy(
        "operation",
        "subscription_status"
    )
    .count()
    .orderBy(
        "operation",
        "subscription_status"
    )
)

display(
    subscription_changes_df
    .orderBy(
        "event_timestamp",
        "subscription_id"
    )
    .limit(20)
)

## 6. Persist and Revalidate the Raw Subscription CDC Batch

Persist the validated subscription CDC events as raw JSON files and verify the stored event and operation counts.

In [0]:
# The synthetic subscription CDC batch is fully regenerated on every run.
(
    subscription_changes_df.write
    .format("json")
    .mode("overwrite")
    .save(SUBSCRIPTIONS_CHANGE_BATCH_PATH)
)

saved_subscription_changes_df = (
    spark.read
    .schema(SUBSCRIPTION_SCHEMA)
    .json(SUBSCRIPTIONS_CHANGE_BATCH_PATH)
)

saved_event_count = (
    saved_subscription_changes_df.count()
)

saved_operation_counts = {
    row["operation"]: row["count"]
    for row in (
        saved_subscription_changes_df
        .groupBy("operation")
        .count()
        .collect()
    )
}

assert saved_event_count == EXPECTED_EVENT_COUNT

assert (
    saved_operation_counts.get("UPDATE")
    == EXPECTED_UPDATE_COUNT
)

assert (
    saved_operation_counts.get("INSERT")
    == INSERT_COUNT
)

print(
    f"Saved subscription events: "
    f"{saved_event_count:,}"
)

print(
    f"Target path: "
    f"{SUBSCRIPTIONS_CHANGE_BATCH_PATH}"
)

display(
    saved_subscription_changes_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)